In [1]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
import random
import pygame
from loguru import logger
import json
import os

# --- Constants ---
SCREEN_WIDTH = 1200
SCREEN_HEIGHT = 800
FPS = 60

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (220, 50, 50)
GREEN = (50, 220, 50)
BLUE = (50, 50, 220)
GRAY = (150, 150, 150)


In [2]:
import gymnasium as gym
from gymnasium.spaces import Box
import numpy as np
import pygame
import pygame.gfxdraw
import math
import random
import os
import json
import time
from collections import deque

# --- Colors & Style ---
COLORS = {
    'bg': (250, 250, 250),
    'grid': (220, 220, 220),
    'robot': (231, 76, 60),      # Red
    'hand': (46, 204, 113),      # Green
    'arm_safe': (200, 200, 200), # Grey (Passable)
    'arm_block': (230, 126, 34), # Orange (Blocked)
    'trajectory': (52, 152, 219),# Blue
    'text': (50, 60, 80)
}

class CustomEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}

    def __init__(self, render_mode=None):
        super().__init__()
        
        # --- Dimensions & Physics ---
        self.grid_size = 10
        self.cell_size = 50 
        self.env_width = self.grid_size * 1.5
        self.env_height = self.grid_size
        self.margin = 0.3
        
        # --- Arm Physics ---
        self.max_length_arm = self.grid_size * 0.8 # 手臂最大长度
        self.arm_blocking_ratio = 0.3  # 手臂靠近手掌的 70% 是实体的，30% 是虚的
        self.fixed_point = np.array([self.env_width / 2, self.env_height]) # 初始固定点
        self.arm_blocking_length = 2

        # --- Thresholds & Rewards ---
        self.distance_threshold_penalty = 5
        self.distance_threshold_collision = 2
        self.distance_threshold_arm = 1 # 碰到手臂线的判定距离
        
        self.reward_step = 2
        self.reward_hand = -100.0       # 被手抓到的惩罚
        self.reward_arm = -100.0         # 撞到手臂的惩罚
        self.reward_bound = -200      # 出界惩罚
        self.reward_max_step = 100.0    # 存活奖励
        
        self.penalty_factor = 2.0
        self.distance_reward_factor = 5.0
        self.smooth_action_penalty = 0.5
        
        # --- Movement Parameters ---
        self.stride_robot_random = [1, 2.5]
        self.stride_hand_random = [0.5, 1.5]
        self.hand_move_epsilon = 0.1 # 10% 概率随机移动
        
        self.max_steps = 50
        self.render_mode = render_mode
        self.pause = False
        self.window = None
        self.clock = None



        # ---Hand history---
        self.history_length = 8
        self.hand_history_buffer = deque(maxlen=self.history_length)
        
        # --- Action Space (dx, dy) ---
        self.action_space = Box(low=-1, high=1, shape=(2,), dtype=np.float32)

        # --- Observation Space ---
        # 1. Robot Pos (2)
        # 2. Hand Pos (2)
        # 3. Hand history (16) (History representation)
        # 4. Distance to Hand (1)
        # 5. Boundary Distances (4) [Left, Right, Top, Bottom]
        # 6. Stride Robot (1)
        # 7. Fixed Point (2)
        # 8. Blocking Point (2)
        # Total = 30
        self.observation_shape = 30
    
        self.observation_space = Box(low=-np.inf, high=np.inf, shape=(self.observation_shape,), dtype=np.float32)

        # Internals
        self.random = True
        self.robot_position = np.zeros(2)
        self.hand_position = np.zeros(2)
        self.last_hand_move = np.zeros(2) # For history
        self.last_action = np.zeros(2)
        self.trajectory_points = []
        self.steps = 0
        
        # Noise parameters
        self.noise_obs_sigma = 0.05
        self.noise_action_sigma = 0.05

    def safe_normalize(self, v):
        """防止除零的归一化函数"""
        norm = np.linalg.norm(v)
        if norm < 1e-8:
            return np.zeros_like(v)
        return v / norm

    def _check_line_intersection(self, p1, p2, p3, p4):
        """
        判断线段 p1-p2 (Robot轨迹) 和 p3-p4 (手臂阻挡部分) 是否相交
        使用向量积 (CCW) 算法
        """
        def ccw(A, B, C):
            return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])
        return ccw(p1, p3, p4) != ccw(p2, p3, p4) and ccw(p1, p2, p3) != ccw(p1, p2, p4)

    def calculate_fix_point(self):
        """更新固定点(肩膀/手肘)位置，确保手臂不被拉断"""
        # 假设固定点只能在底边滑动 (y = env_height) 或者根据手的位置动态调整
        # 这里沿用你之前的逻辑：根据手的位置计算固定点的 X，保持 Y 不变（除非你要移动肩膀）
        
        # 简单的反向运动学约束：
        dy = self.fixed_point[1] - self.hand_position[1]
        
        # 如果垂直距离已经超过臂长，限制手的位置（物理约束）
        if abs(dy) > self.max_length_arm:
            # 这种情况通常不应该发生，如果发生了说明手跑太远了
            pass 
        
        # 计算水平距离限制
        term = self.max_length_arm**2 - dy**2
        if term < 0: term = 0
        max_dx = np.sqrt(term)
        
        # 这里的逻辑是：如果手跑远了，肩膀(fixed_point)跟着动
        # 如果你希望肩膀不动，那就应该在 move_hand 里限制手的位置
        # 下面这个逻辑是让肩膀跟着手跑，保持臂长合理
        current_dx = self.fixed_point[0] - self.hand_position[0]
        
        if abs(current_dx) > max_dx:
            if current_dx > 0:
                self.fixed_point[0] = self.hand_position[0] + max_dx
            else:
                self.fixed_point[0] = self.hand_position[0] - max_dx
        
        # 确保 fixed_point 不跑出画面太远
        self.fixed_point[0] = np.clip(self.fixed_point[0], 0, self.env_width)

    def _get_obs(self):
        # 计算 Blocking Point (手臂实体的终点)
        # vec_arm = self.fixed_point - self.hand_position
        # blocking_point = self.hand_position + vec_arm * self.arm_blocking_ratio
        
        # 边界距离
        dist_bounds = np.array([
            self.robot_position[0],                  # Dist Left
            self.env_width - self.robot_position[0], # Dist Right
            self.robot_position[1],                  # Dist Top
            self.env_height - self.robot_position[1] # Dist Bottom
        ])

        obs = np.concatenate((
            self.robot_position,    # [0:2]
            self.hand_position,     # [2:4]
            *self.hand_history_buffer,
            [self.current_distance],# [6]
            dist_bounds,            # [7:11] (Boundary 4)
            [self.stride_robot],    # [11]
            self.fixed_point,       # [12:14]
            self.blocking_point          # [14:16]
        )).astype(np.float32)

        return obs

    def _get_info(self):
        return {
            "distance_to_hand": self.current_distance,
            "robot_position": self.robot_position,
            "hand_position": self.hand_position,
            "steps": self.steps,
            "fixed_point":self.fixed_point
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Randomize parameters
        self.stride_robot = np.random.uniform(*self.stride_robot_random)
        self.stride_hand = np.random.uniform(*self.stride_hand_random)
        
        # Initialize positions
        self.robot_position = np.random.uniform(self.margin, [self.env_width-self.margin, self.env_height-self.margin])
        self.hand_position = np.random.uniform(self.margin, [self.env_width-self.margin, self.env_height-self.margin])
        self.fixed_point = np.array([self.env_width/2, self.env_height]) # Reset Shoulder
        
        self.trajectory_points = [self.robot_position.copy()]
        self.current_distance = np.linalg.norm(self.robot_position - self.hand_position)
        self.pre_distance = self.current_distance
        self.last_action = np.zeros(2)
        self.last_hand_move = np.zeros(2)
        self.steps = 0
        self.distance = [self.current_distance]
        
        # Initial Physics Update
        self.calculate_fix_point()

        self.hand_history_buffer.clear()
        for i in range(self.history_length):
            self.hand_history_buffer.append(np.zeros(2))

        return self._get_obs(),self._get_info()

    def _reward(self,action):

        reward = 0
        terminated = False
        truncated = False
        done_reason = None
        
        self.steps += 1
        
        # 1. Boundary Check
        if np.any(self.robot_position <= self.margin) or \
           self.robot_position[0] >= self.env_width - self.margin or \
           self.robot_position[1] >= self.env_height - self.margin:
            reward += self.reward_bound
            terminated = True
            done_reason = "Out of Bounds"

        # 2. Hand Collision Check
        self.current_distance = np.linalg.norm(self.robot_position - self.hand_position)
        self.distance.append(self.current_distance)
        
        if self.current_distance < self.distance_threshold_collision:
            reward += self.reward_hand
            terminated = True
            done_reason = "Caught by Hand"
        elif self.current_distance < self.distance_threshold_penalty:
            # 靠近惩罚 (Distance Penalty)
            reward -= self.penalty_factor * (self.distance_threshold_penalty - self.current_distance)

        # 3. Arm Collision Check (Line Intersection)

        
        # 检测 Robot 轨迹是否穿过 "手掌 -> BlockingPoint" 这一段
        if self._check_line_intersection(self.old_robot_pos, self.robot_position, self.hand_position, self.blocking_point):
            reward += self.reward_arm
            terminated = True
            done_reason = "Hit Arm"

        # 4. Reward Shaping (Distance Improvement)
        reward += (self.pre_distance - self.current_distance) * self.distance_reward_factor
        self.pre_distance = self.current_distance
        
        
        
        # 6. Step Reward
        reward += self.reward_step
        
        # 7. Max Steps
        if self.steps >= self.max_steps:
            truncated = True
            reward += self.reward_max_step

        return reward,terminated,truncated,done_reason
    


    def _get_hand_movement(self):
        """计算手掌移动向量"""
        if random.random() < self.hand_move_epsilon:
            # 随机移动
            move = np.random.uniform(-1, 1, size=2)
            move = self.safe_normalize(move) * self.stride_hand
        else:
            # 追逐机器人
            dir_vector = self.robot_position - self.hand_position
            move = self.safe_normalize(dir_vector) * self.stride_hand
            
        return move

 
    def step(self, action):

        if self.random:
            action+=np.random.normal(0,self.noise_action_sigma,size=self.action_space.shape)  # Add some noise to action to make it more realistic

        move_hand = self._get_hand_movement()
        self.hand_position += move_hand  # Update hand position
        self.hand_position = np.clip(self.hand_position, self.margin, [self.env_width-self.margin,self.env_height-self.margin])  # Ensure hand stays within grid bounds
        self.hand_history_buffer.append(move_hand)

        self.old_robot_pos = self.robot_position.copy()
        self.robot_position += action * self.stride_robot  # Scale the action to control speed
        self.trajectory_points.append(self.robot_position.copy()) # New: Add current position to trajectory

        self.calculate_fix_point()  # Update fixed point based on new hand position
        vec_arm = self.fixed_point - self.hand_position
        unit_vec_arm = self.safe_normalize(vec_arm)
        self.blocking_point = self.hand_position + unit_vec_arm * self.arm_blocking_length
        if self.blocking_point[1]>self.env_height:
            self.blocking_point = self.fixed_point.copy()
        self.steps += 1
        

        reward,terminated,truncated,done_reason = self._reward(action)
        info = self._get_info()
        info['done_reason'] = done_reason
        info['distance_mean'] = np.mean(self.distance)
        observation = self._get_obs()
        if self.random:
            observation += np.random.normal(0, self.noise_obs_sigma, size=self.observation_shape)  # Add some noise to observation to make it more realistic

        return observation, reward, terminated, truncated, info

    # --- Rendering Functions ---

    def _draw_capsule(self, surf, color, start, end, width):
        """Helper to draw a capsule"""
        p1 = np.array(start)
        p2 = np.array(end)
        length = np.linalg.norm(p2 - p1)
        if length < 1e-6: return
        
        # Angle
        angle = math.atan2(p2[1] - p1[1], p2[0] - p1[0])
        sin_a, cos_a = math.sin(angle), math.cos(angle)
        
        dx = (width / 2) * sin_a
        dy = (width / 2) * cos_a
        
        points = [
            (p1[0] - dx, p1[1] + dy),
            (p2[0] - dx, p2[1] + dy),
            (p2[0] + dx, p2[1] - dy),
            (p1[0] + dx, p1[1] - dy)
        ]
        
        # Convert to screen coords (assuming input is pixels)
        points_px = [(int(x), int(y)) for x, y in points]
        
        pygame.gfxdraw.aapolygon(surf, points_px, color)
        pygame.gfxdraw.filled_polygon(surf, points_px, color)
        
        # Ends
        pygame.gfxdraw.aacircle(surf, int(p1[0]), int(p1[1]), int(width/2), color)
        pygame.gfxdraw.filled_circle(surf, int(p1[0]), int(p1[1]), int(width/2), color)
        pygame.gfxdraw.aacircle(surf, int(p2[0]), int(p2[1]), int(width/2), color)
        pygame.gfxdraw.filled_circle(surf, int(p2[0]), int(p2[1]), int(width/2), color)

    def render(self, mode="human"):
        if self.window is None:
            pygame.init()
            self.window = pygame.display.set_mode(
                (int(self.env_width * self.cell_size), int(self.env_height * self.cell_size))
            )
            pygame.display.set_caption("Rehabilitation Robot Env")
            self.clock = pygame.time.Clock()

        # Handle Events
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.close()
                return

        # Canvas
        canvas = pygame.Surface((self.window.get_width(), self.window.get_height()))
        canvas.fill(COLORS['bg'])

        # Draw Grid
        for x in range(0, int(self.env_width * self.cell_size), self.cell_size):
            pygame.draw.line(canvas, COLORS['grid'], (x, 0), (x, canvas.get_height()))
        for y in range(0, int(self.env_height * self.cell_size), self.cell_size):
            pygame.draw.line(canvas, COLORS['grid'], (0, y), (canvas.get_width(), y))

        # Convert Coords to Pixels
        to_px = lambda pos: (int(pos[0] * self.cell_size), int(pos[1] * self.cell_size))
        
        hand_px = to_px(self.hand_position)
        fix_px = to_px(self.fixed_point)
        robot_px = to_px(self.robot_position)
        
        # Calculate Blocking Point for Render
        vec_arm = self.fixed_point - self.hand_position
        blocking_point = self.hand_position + vec_arm * self.arm_blocking_ratio
        block_px = to_px(blocking_point)

        # 1. Draw Trajectory
        if len(self.trajectory_points) > 1:
            traj_px = [to_px(p) for p in self.trajectory_points[-50:]] # Last 50 points
            if len(traj_px) > 1:
                pygame.draw.lines(canvas, COLORS['trajectory'], False, traj_px, 2)
                for p in traj_px:
                     pygame.draw.circle(canvas, COLORS['trajectory'], p, 2)

        # 2. Draw Arm (Safe Zone - Elbow to Block)
        self._draw_capsule(canvas, COLORS['arm_safe'], block_px, fix_px, 15)
        
        # 3. Draw Arm (Block Zone - Hand to Block)
        self._draw_capsule(canvas, COLORS['arm_block'], hand_px, block_px, 20)

        # 4. Draw Hand (Circle representation for now, or use image)
        pygame.gfxdraw.aacircle(canvas, hand_px[0], hand_px[1], int(self.cell_size * 0.3), COLORS['hand'])
        pygame.gfxdraw.filled_circle(canvas, hand_px[0], hand_px[1], int(self.cell_size * 0.3), COLORS['hand'])
        
        # 5. Draw Robot
        pygame.gfxdraw.aacircle(canvas, robot_px[0], robot_px[1], int(self.cell_size * 0.25), COLORS['robot'])
        pygame.gfxdraw.filled_circle(canvas, robot_px[0], robot_px[1], int(self.cell_size * 0.25), COLORS['robot'])
        # Highlight
        pygame.gfxdraw.filled_circle(canvas, robot_px[0]-3, robot_px[1]-3, int(self.cell_size * 0.08), (255, 255, 255))

        # Flip
        self.window.blit(canvas, (0, 0))
        pygame.display.flip()
        self.clock.tick(self.metadata["render_fps"])

    def save_args(self, path):
        env_args = {
            "grid_size": self.grid_size,
            "max_steps": self.max_steps,
            "arm_blocking_ratio": self.arm_blocking_ratio
        }
        os.makedirs(path, exist_ok=True)
        with open(os.path.join(path, "env_args.json"), "w") as f:
            json.dump(env_args, f, indent=4)

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()

In [66]:
import numpy as np
from collections import deque

class BiomechanicalFilter:
    def __init__(self, mode='healthy'):
        """
        初始化滤波器
        :param mode: 'healthy', 'parkinson', 'stroke', 'ataxia'
        :param dt: 仿真步长 (秒)
        """
        self.mode = mode
        self.t = 0
        
        # --- 帕金森参数 (基于 Rocon et al. 2004) ---
        self.tremor_amp = 0.4     # 震颤幅度 (像素或单位距离)
        self.tremor_freq = 1/8    # 频率 Hz (4-6Hz 是典型值)
        
        # --- 中风参数 (基于 Rohrer et al. 2002) ---
        self.drag_factor = 0.6    # 肌无力: 只能发挥 60% 的速度
        self.submove_timer = 0    # 子运动计时器
        self.is_stuck = False     # 是否处于"停顿"状态
        
        # --- 共济失调参数 (基于 Manto et al. 1994) ---
        self.dysmetria_gain = 1.5 # 过冲系数 (意向性震颤)
        self.momentum = np.zeros(2) # 惯性/动量
        self.friction = 0.1       # 模拟刹不住车的摩擦系数
        
        # --- 通用: 延迟缓冲区 ---
        self.delay_buffer = deque(maxlen=10) # 模拟反应延迟

    def reset(self):
        self.t = 0
        self.momentum = np.zeros(2)
        self.delay_buffer.clear()
        self.is_stuck = False

    def apply(self, ideal_action, current_pos, target_pos=None):
        """
        核心函数
        :param ideal_action: (dx, dy) 原始动作，假设范围 [-1, 1] 或速度向量
        :param current_pos: 当前手的位置 (x, y)
        :param target_pos: 目标位置 (用于共济失调计算距离)，可选
        :return: (dx, dy) 实际执行的动作
        """
        self.t += 1
        
        # 1. 基础处理：将其转换为 numpy 数组
        action = np.array(ideal_action, dtype=float)

        # 2. 根据模式分发处理
        if self.mode == 'healthy':
            final_action = self._apply_healthy(action)
            
        elif self.mode == 'parkinson':
            final_action = self._apply_parkinson(action)
            
        elif self.mode == 'stroke':
            final_action = self._apply_stroke(action)
            
        elif self.mode == 'ataxia':
            final_action = self._apply_ataxia(action, current_pos, target_pos)
            
        else:
            final_action = action

        # 3. 全局物理约束 (Sim2Real 保护)
        # 防止瞬间速度过大导致物理引擎穿模
        final_action = np.clip(final_action, -10, 10) # 假设最大速度限制
        
        return final_action

    # ---------------- 具体病理实现 ----------------

    def _apply_healthy(self, action):
        # 正常人也有微小的 Minimum Jerk 平滑，这里简化为不做改动
        # 或者加极微小的高斯噪声模拟传感器误差
        return action

    def _apply_parkinson(self, action):
        """
        模型依据: Rocon et al. (2004)
        叠加正弦震颤 + 随机高斯噪声
        """
        # 计算震颤向量
        tremor_x = self.tremor_amp * np.sin(2 * np.pi * self.tremor_freq * self.t)
        tremor_y = self.tremor_amp * np.cos(2 * np.pi * self.tremor_freq * self.t)
        
        # 随机相位漂移 (让震颤看起来不那么机械)
        noise = np.random.normal(0, 0.2, 2)
        
        return action + np.array([tremor_x, tremor_y]) + noise

    def _apply_stroke(self, action):
        """
        模型依据: Rohrer et al. (2002) - Submovements
        表现为: 迟缓 (Lag) + 间歇性停顿 (Stuttering) + 速度削弱
        """
        # 1. 模拟神经延迟 (Lag)
        self.delay_buffer.append(action)
        if len(self.delay_buffer) < 4: # 假设延迟 3 帧
            return np.zeros(2)
        delayed_action = self.delay_buffer.popleft()
        
        # 2. 模拟子运动 (间歇性停顿)
        # 每隔一段时间随机"卡住"一下
        if self.submove_timer <= 0:
            if np.random.rand() < 0.1: # 10% 概率进入停顿
                self.is_stuck = True
                self.submove_timer = np.random.randint(2, 5) # 停顿 5-15 帧
            else:
                self.is_stuck = False
                self.submove_timer = np.random.randint(5, 10) # 正常运动 20-60 帧
        
        self.submove_timer -= 1
        
        if self.is_stuck:
            return delayed_action * 0.1 # 几乎不动
        else:
            # 3. 肌无力 (Weakness)
            return delayed_action * self.drag_factor

    def _apply_ataxia(self, action, current_pos, target_pos):
        """
        模型依据: Manto et al. (1994) - Dysmetria
        表现为: 惯性过大 (刹不住车) + 距离相关的抖动
        """
        # 1. 意向性震颤 (距离目标越近，抖动越大)
        dist_noise = 0
        if target_pos is not None:
            dist = np.linalg.norm(current_pos - target_pos)
            # 距离越近(dist小)，噪声因子反而需要处理，
            # 这里简化为：速度增益误差
        
        # 2. 动量模型 (High Momentum / Low Friction)
        # 新的速度不是直接等于 Action，而是很大程度上受上一步速度影响
        # V_new = V_old * (1 - friction) + Action * Gain
        
        self.momentum = self.momentum * (1 - self.friction) + action * self.dysmetria_gain
        
        return self.momentum
    def _apply_attention(self):
        pass


In [62]:
import gymnasium as gym
from gymnasium.spaces import Box,Dict
import numpy as np
import pygame
import pygame.gfxdraw
import math
import random
import os
import json
import time
from collections import deque



# --- Main Environment Class ---
class RehabilitationEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 60}

    def __init__(self, 
                 training_mode='robot', # 'robot' or 'hand'
                 robot_model=None,      # 训练 Hand 时需要的 Robot 对手
                 hand_model=None,       # [新增] 训练 Robot 时需要的 Hand 对手
                 hand_type='healthy',   # 'healthy', 'parkinson', 'stroke'
                 render_mode=None):
        
        super().__init__()
        self.training_mode = training_mode
        self.robot_model = robot_model
        self.hand_model_record = hand_model    # [新增] 保存手掌模型
        self.hand_type = hand_type
        self.render_mode = render_mode
        
        # --- Dimensions ---
        self.grid_size = 10
        self.cell_size = 50 
        self.env_width = self.grid_size * 1.5
        self.env_height = self.grid_size
        self.margin = 0.3
        
        # --- Physics & Arm ---
        self.max_length_arm = self.grid_size * 0.9
        self.fixed_point = np.array([self.env_width / 2, self.env_height])
        self.arm_blocking_length = 4
        
        # --- Filters ---
        self.bio_filter = BiomechanicalFilter(self.hand_type)
        
        # --- Thresholds ---
        self.distance_threshold_collision = 3
        self.distance_threshold_penalty = 3.0
        
        # --- Rewards Config ---
        self.reward_hand_catch = 100
        self.reward_robot_caught = -100
        self.reward_arm_hit = -100
        self.reward_bound = -200
        self.reward_step = -2 if training_mode == 'hand' else 2
        self.reward_survival = 100
        
        # Biomechanical Cost Weights
        self.w_sweep = 10.0
        self.w_effort = 2.0
        
        # --- Movement Params ---
        self.stride_robot_random = [1, 2.5]
        self.stride_hand_random = [0.5, 1.5]
        self.hand_move_epsilon = 0.1
        
        self.max_steps = 50
        self.steps = 0
        
        # --- Spaces ---
        self.action_space = Box(low=-1, high=1, shape=(2,), dtype=np.float32)

        # Observation space definition
        self.history_length = 8
        # obs dim calculation: 
        # robot(2) + hand(2) + history(20) + dist(1) + bounds(4) + stride(1) + fix(2) + block(2) = 34
        self.obs_dim = 30
        self.observation_space = Box(low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32)

        # --- Internals ---
        self.robot_position = np.zeros(2)
        self.hand_position = np.zeros(2)
        self.blocking_point = np.zeros(2)
        self.hand_history_buffer = deque(maxlen=self.history_length)
        self.trajectory_points = []
        
        self.window = None
        self.clock = None
        self.random_noise = True
        self.noise_sigma = 0.05

    def safe_normalize(self, v):
        norm = np.linalg.norm(v)
        if norm < 1e-8: return np.zeros_like(v)
        return v / norm

    def _check_line_intersection(self, p1, p2, p3, p4):
        """判断线段相交"""
        def ccw(A, B, C):
            return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])
        return ccw(p1, p3, p4) != ccw(p2, p3, p4) and ccw(p1, p2, p3) != ccw(p1, p2, p4)

    def _calculate_fix_point(self):
        """简单反向运动学"""
        vec = self.hand_position - self.fixed_point
        dist = np.linalg.norm(vec)
        if dist > self.max_length_arm:
            dx = abs(self.hand_position[0] - self.fixed_point[0])
            dy = abs(self.hand_position[1] - self.fixed_point[1])
            max_dx = np.sqrt(max(0, self.max_length_arm**2 - dy**2))
            target_x = self.hand_position[0] - max_dx if self.hand_position[0] < self.fixed_point[0] else self.hand_position[0] + max_dx
            self.fixed_point[0] = 0.8 * self.fixed_point[0] + 0.2 * target_x
            self.fixed_point[0] = np.clip(self.fixed_point[0], 0, self.env_width)


    def _calculate_biomechanical_cost(self, current_pos, move_vec):
        """计算生物力学成本 (用于训练Hand Agent)"""
        arm_vec = current_pos - self.fixed_point
        arm_len = np.linalg.norm(arm_vec)
        if arm_len < 1e-4: return 0
        arm_dir = arm_vec / arm_len
        v_rad = np.dot(move_vec, arm_dir) * arm_dir # 径向速度(伸缩)
        v_tan = move_vec - v_rad # 切向速度(扫动)
        s_tan = np.linalg.norm(v_tan)
        s_rad = np.linalg.norm(v_rad)
        # 惩罚远端的大幅度扫动 (Prevent full-screen sweep)
        p_sweep = self.w_sweep * (s_tan**2) * (1 + 2.0 * (arm_len / self.max_length_arm)**2)
        p_effort = self.w_effort * (s_rad**2)
        return p_sweep + p_effort

    def _get_obs(self):
        dist_bounds = np.array([
            self.robot_position[0],
            self.env_width - self.robot_position[0],
            self.robot_position[1],
            self.env_height - self.robot_position[1]
        ])
        
        flat_history = np.array(self.hand_history_buffer).flatten()
        if len(flat_history) < self.history_length * 2:
            flat_history = np.zeros(self.history_length * 2)

        obs = np.concatenate((
            self.robot_position,    
            self.hand_position,     
            [self.current_distance],
            dist_bounds,            
            [self.stride_robot],    
            self.fixed_point,       
            self.blocking_point ,
            flat_history,           

        )).astype(np.float32)
        return obs

    def _get_info(self):
        return {
            "dist": self.current_distance,
            "steps": self.steps,
            "hand_pos": self.hand_position,
            "robot_pos": self.robot_position,
            "fixed_point": self.fixed_point,
            "blocking_point": self.blocking_point,
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.bio_filter.mode = random.choice(['healthy', 'parkinson','stroke']) 
        self.hand_model =  random.choice([None, self.hand_model_record]) 

        self.stride_robot = np.random.uniform(*self.stride_robot_random)
        self.stride_hand = np.random.uniform(*self.stride_hand_random)
        
        self.robot_position = np.random.uniform(self.margin, [self.env_width-self.margin, self.env_height-self.margin])
        self.hand_position = np.random.uniform(self.margin, [self.env_width-self.margin, self.env_height-self.margin])
        self.fixed_point = np.array([self.env_width/2, self.env_height])
        
        self.hand_history_buffer.clear()
        for _ in range(self.history_length):
            self.hand_history_buffer.append(np.zeros(2))
            
        self.trajectory_points = [self.robot_position.copy()]
        self.current_distance = np.linalg.norm(self.robot_position - self.hand_position)
        self.pre_distance = self.current_distance
        self.steps = 0
        
        self._calculate_fix_point()
        vec = self.hand_position - self.fixed_point
        self.blocking_point = self.hand_position # 默认blocking point等于手
        
        return self._get_obs(), self._get_info()

    def _get_scripted_hand_move(self):
        """后备的脚本控制"""
        if random.random() < self.hand_move_epsilon:
            move = np.random.uniform(-1, 1, size=2)
            move = self.safe_normalize(move) * self.stride_hand
        else:
            vec = self.robot_position - self.hand_position
            move = self.safe_normalize(vec) * self.stride_hand
        return move

    def step(self, action):
        # 1. 预处理
        robot_move = np.zeros(2)
        hand_move_raw = np.zeros(2)
        bio_cost = 0.0
        
        if self.random_noise:
            action += np.random.normal(0, self.noise_sigma, size=action.shape)

        # 2. 决策逻辑 (关键修改部分)
        if self.training_mode == 'robot':
            # --- 训练 Robot: Robot 使用 Action, Hand 使用 Model 或 Script ---
            robot_move = action * self.stride_robot
            
            # [修改点] 检查是否有预训练的手掌模型
            if self.hand_model is not None:
                # 获取观测 (Hand Agent 也是用同样的 Env Observation)
                obs_for_hand = self._get_obs()
                # 预测动作
                hand_action, _ = self.hand_model.predict(obs_for_hand, deterministic=False) # False 保持一定的随机探索性
                hand_move_raw = hand_action * self.stride_hand
            else:
                # 如果没有模型，回退到脚本
                hand_move_raw = self._get_scripted_hand_move()
                
        else:
            # --- 训练 Hand: Hand 使用 Action, Robot 使用 Model ---
            hand_move_raw = action * self.stride_hand
            
            if self.robot_model is not None:
                obs_for_robot = self._get_obs() 
                robot_action, _ = self.robot_model.predict(obs_for_robot, deterministic=True)
                robot_move = robot_action * self.stride_robot
            else:
                robot_move = np.zeros(2) 

        # 3. 动作后处理 (Apply Filter)
        # 无论手是脚本控制还是模型控制，都要经过生物力学滤波器（模拟病理身体）
        if self.training_mode == 'hand':
            bio_cost = self._calculate_biomechanical_cost(self.hand_position, hand_move_raw)
            
        hand_move_final = self.bio_filter.apply(hand_move_raw, self.hand_position)

        # 4. 物理更新
        self.hand_position += hand_move_final
        self.hand_position = np.clip(self.hand_position, self.margin, [self.env_width-self.margin, self.env_height-self.margin])
        self.hand_history_buffer.append(hand_move_final)
        
        old_robot_pos = self.robot_position.copy()
        self.robot_position += robot_move
        self.trajectory_points.append(self.robot_position.copy())
        
        self._calculate_fix_point()
        vec_arm = self.fixed_point - self.hand_position
        dist_arm = np.linalg.norm(vec_arm)
        to_shoulder = self.safe_normalize(vec_arm)
        self.blocking_point = self.hand_position + to_shoulder * min(self.arm_blocking_length, dist_arm)

        self.steps += 1
        
        # 5. 奖励计算
        reward = 0
        terminated = False
        truncated = False
        done_reason = None
        
        self.current_distance = np.linalg.norm(self.robot_position - self.hand_position)
        
        # Robot Out of Bounds
        if np.any(self.robot_position <= self.margin) or \
           self.robot_position[0] >= self.env_width - self.margin or \
           self.robot_position[1] >= self.env_height - self.margin:
            if self.training_mode == 'robot': reward += self.reward_bound
            terminated = True
            done_reason = "Robot Out"

        # Caught
        if self.current_distance < self.distance_threshold_collision:
            if self.training_mode == 'robot': 
                reward += self.reward_robot_caught
            else: 
                reward += self.reward_hand_catch
            terminated = True
            done_reason = "Robot Caught"
            
        # Hit Arm
        if self._check_line_intersection(old_robot_pos, self.robot_position, self.hand_position, self.blocking_point):
            if self.training_mode == 'robot':
                reward += self.reward_arm_hit
            terminated = True
            done_reason = "Hit Arm"

        # Shaping
        dist_improvement = self.pre_distance - self.current_distance
        if self.training_mode == 'robot':
            reward += self.reward_step
            reward -= dist_improvement * 2
        else:
            reward += dist_improvement * 5.0
            reward += self.reward_step
            reward -= bio_cost

        self.pre_distance = self.current_distance

        if self.steps >= self.max_steps:
            truncated = True
            if self.training_mode == 'robot': reward += self.reward_survival

        obs = self._get_obs()
        if self.random_noise:
            obs += np.random.normal(0, self.noise_sigma, size=obs.shape)
            
        info = self._get_info()
        info['bio_cost'] = bio_cost
        info['done_reason'] = done_reason
        
        return obs, reward, terminated, truncated, info

    # --- Render (保持不变或微调) ---
    def render(self, mode="human"):
        if self.window is None:
            pygame.init()
            self.window = pygame.display.set_mode(
                (int(self.env_width * self.cell_size), int(self.env_height * self.cell_size))
            )
            pygame.display.set_caption(f"Mode: {self.training_mode.upper()} | Hand: {self.hand_type}")
            self.clock = pygame.time.Clock()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.close()

        canvas = pygame.Surface((self.window.get_width(), self.window.get_height()))
        canvas.fill(COLORS['bg'])
        
        to_px = lambda p: (int(p[0]*self.cell_size), int(p[1]*self.cell_size))

        hand_px = to_px(self.hand_position)
        robot_px = to_px(self.robot_position)
        fix_px = to_px(self.fixed_point)
        block_px = to_px(self.blocking_point)

        # Draw trajectory
        if len(self.trajectory_points) > 1:
            pts = [to_px(p) for p in self.trajectory_points[-50:]]
            if len(pts) > 1:
                pygame.draw.lines(canvas, COLORS['trajectory'], False, pts, 2)

        # Draw Arm
        pygame.draw.line(canvas, COLORS['arm_safe'], fix_px, block_px, 5)
        pygame.draw.line(canvas, COLORS['arm_block'], block_px, hand_px, 15)
        
        # Draw Entities
        pygame.draw.circle(canvas, (50,50,50), fix_px, 8) # Shoulder
        pygame.draw.circle(canvas, COLORS['hand'], hand_px, int(self.cell_size*0.3)) # Hand
        pygame.draw.circle(canvas, COLORS['robot'], robot_px, int(self.cell_size*0.25)) # Robot

        self.window.blit(canvas, (0,0))
        pygame.display.flip()
        self.clock.tick(self.metadata["render_fps"])

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()

In [5]:
import gymnasium as gym
from gymnasium.spaces import Box, Discrete,Tuple
import numpy as np
import pygame

# Define colors
WHITE = (255, 255, 255)
RED = (255, 0, 0)
GREEN = (0, 255, 0)
BLUE = (0, 0, 255)  # Color for the trajectory

class CustomEnv_hand(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 60}
    hand_mode : str   #  'healthy', 'parkinson', 'stroke', 'ataxia'
    def __init__(self,robot_model,render_mode=None,hand_mode = 'healthy'):
        super().__init__()
        if hand_mode not in ['healthy', 'parkinson', 'stroke', 'ataxia']:
            raise ValueError("Invalid hand mode,must in ['healthy', 'parkinson', 'stroke', 'ataxia']")
        self.grid_size = 10
        self.robot_model = robot_model
        self.last_action_robot = np.zeros(2)
        self.env_width = self.grid_size*1.5
        self.env_height = self.grid_size
        

        self.pause = False
        self.domain_randomization = False
        self.render_mode = render_mode

        # Define two separate thresholds for obstacle handling
        self.distance_threshold_penalty = 5  # Penalty zone threshold (larger value)
        self.distance_threshold_collision = 1.5  # Collision threshold (smaller value)
        self.distance_threshold_arm = 1  # Arm threshold (smaller value)
        self.penalty_factor = -5  # Penalty scaling factor
        self.distance_reward_factor = -2
        self.smooth_action_penalty = -5
        self.steps = 0
        self.margin = 0.3
        self.reward_arm =0
        self.reward_hand = 300
        self.reward_bound = -400
        self.reward_max_step = 0
        self.reward_step = -10
        self.stride_robot_random = [1,3]
        self.stride_hand_random = [0.6,1]
        self.hand_move_epsilon = 0.1

        self.hand_history_buffer = deque(maxlen=8)
        self.arm_blocking_length = 2
        self.blocking_point = np.array([0,0])


        self.current_distance = 0  # Current distance to goal, used for reward shaping
        self.max_steps = 50  # Set a maximum number of steps to prevent infinite loops
        # Action space (dx, dy)
        self.action_space = Box(low=-1, high=1, shape=(2,), dtype=np.float32)
        # Observation space (robot_x, robot_y, goal_x, goal_y)
        self.observation_shape = 2+2+2+1+1+1+2+1+1+2+2 # Robot position, hand position, velocity_hand,radius_hand, and distance to hand

        self.observation_space = Box(low=0, high=np.array([self.stride_hand_random[1],0.5,1,self.env_width,self.env_height, self.env_width, self.env_height,1,1 , (2**0.5)*self.grid_size,0.5*self.grid_size,0.5*self.grid_size,2*self.grid_size,self.grid_size,self.stride_robot_random[1],self.env_width,self.env_height]), 
                                     shape=(self.observation_shape,), dtype=np.float32)

        self.random = True
        # For rendering
        self.window = None
        self.clock = None
        self.cell_size = 50 # Pixels per grid unit
        self.trajectory_points = [] # New: List to store past robot positions
        self.dist_arm = 0
        self.max_length_arm = self.grid_size

        self.execution_filter = BiomechanicalFilter(hand_mode)

    def calculate_fix_point(self):
        dist_arm = np.linalg.norm(self.fixed_point - self.hand_position)
        # print(dist_arm,self.fixed_point,self.hand_position)

        if dist_arm > self.max_length_arm:
            if self.fixed_point[0] < self.hand_position[0]:
                self.fixed_point[0] = self.hand_position[0] - np.sqrt(self.max_length_arm**2-(self.fixed_point[1]-self.hand_position[1])**2)
            else:
                self.fixed_point[0] = self.hand_position[0] + np.sqrt(self.max_length_arm**2-(self.fixed_point[1]-self.hand_position[1])**2)

        # === 核心改进 2: 生物力学成本 (防平扫) ===
    def _calculate_biomechanical_cost(self, current_pos, proposed_movement):
        """
        计算动作的生物力学成本，惩罚远端的切向运动(平扫)
        """
        # 1. 手臂向量 (Base -> Hand)
        arm_vec = current_pos - self.fixed_point
        arm_len = np.linalg.norm(arm_vec)
        
        if arm_len < 1e-4: return 0
        
        # 2. 动作分解
        arm_dir = arm_vec / arm_len # 单位向量
        
        # 径向速度 (伸缩)
        v_radial_scalar = np.dot(proposed_movement, arm_dir)
        v_radial = v_radial_scalar * arm_dir
        
        # 切向速度 (横扫)
        v_tangential = proposed_movement - v_radial
        
        speed_tan = np.linalg.norm(v_tangential)
        speed_rad = abs(v_radial_scalar)
        
        # 3. 惩罚公式
        # 切向惩罚：随臂长平方增长
        penalty_sweep =  20* (speed_tan**2) * (1 + 2.0 * (arm_len / self.max_length_arm)**2)
        
        # 径向惩罚：很小，允许伸缩
        penalty_effort =  5* (speed_rad**2)
        
        return penalty_sweep + penalty_effort
    
    def safe_normalize(self, v):
        """防止除零的归一化函数"""
        norm = np.linalg.norm(v)
        if norm < 1e-8:
            return np.zeros_like(v)
        return v / norm

    def dist_point_to_segment_correct(self,P, A, B, eps=1e-12):
        P = np.asarray(P, dtype=float)
        A = np.asarray(A, dtype=float)
        B = np.asarray(B, dtype=float)
        v = B - A
        w = P - A
        vv = np.dot(v, v)
        if vv <= eps:
            # A and B coincide: treat as point A
            C = A.copy()
            d = np.linalg.norm(P - A)
            t = 0.0
            case = 'endpoint_A'
        else:
            t = np.dot(w, v) / vv
            if t < 0.0:
                C = A
                d = np.linalg.norm(P - A)
                case = 'before_A'
            elif t > 1.0:
                C = B
                d = np.linalg.norm(P - B)
                case = 'after_B'
            else:
                C = A + t * v
                d = np.linalg.norm(P - C)
                case = 'on_segment'
        return float(d), C, float(t), case

    def _get_obs(self):
        
        return np.concatenate(( [np.array([self.stride_hand])]+
                               [np.array([self.random_epsilon])]+
                               [np.array([self.direction_error])]+
                               
                               
                                [self.robot_position]+ 
                               [self.hand_position]+ 
                               [self.last_action]+
                               [np.array([self.current_distance])]+
                               [np.array([min(self.hand_position[0],
                                              self.hand_position[1],
                                              self.env_width-self.hand_position[0],
                                              self.env_height-self.hand_position[1])])]+
                                [np.array([self.dist_arm])]+
                                [self.fixed_point]+
                                [np.array([self.stride_robot])]+
                                [np.array([self.env_width,self.env_height])]))
    
    def _get_obs_robot(self):
                # 边界距离
        dist_bounds = np.array([
            self.robot_position[0],                  # Dist Left
            self.env_width - self.robot_position[0], # Dist Right
            self.robot_position[1],                  # Dist Top
            self.env_height - self.robot_position[1] # Dist Bottom
        ])

        obs = np.concatenate((
            self.robot_position,    # [0:2]
            self.hand_position,     # [2:4]
            *self.hand_history_buffer,
            [self.current_distance],# [6]
            dist_bounds,            # [7:11] (Boundary 4)
            [self.stride_robot],    # [11]
            self.fixed_point,       # [12:14]
            self.blocking_point          # [14:16]
        )).astype(np.float32)

        return obs
        
    def _check_line_intersection(self, p1, p2, p3, p4):
        """
        判断线段 p1-p2 (Robot轨迹) 和 p3-p4 (手臂阻挡部分) 是否相交
        使用向量积 (CCW) 算法
        """
        def ccw(A, B, C):
            return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])
        return ccw(p1, p3, p4) != ccw(p2, p3, p4) and ccw(p1, p2, p3) != ccw(p1, p2, p4)

    def _get_info(self):
        return {
            "distance_to_hand": self.current_distance,
            "robot_position": self.robot_position,
            "hand_position": self.hand_position,
            'distance_arm':self.dist_arm,
            "fix_point":self.fixed_point,
        }

    def reset(self, seed=None, options=None):

        super().reset()
        self.distance = []
        self.stride_robot = np.random.uniform(*self.stride_robot_random)  # Randomize stride length
        self.stride_hand = np.random.uniform(*self.stride_hand_random)  # Randomize stride length
        # self.stride_robot = 1  # Randomize stride length
        self.distance_threshold_collision = np.random.uniform(1.5,2.5)  # Randomize collision threshold
        self.distance_threshold_penalty = np.random.uniform(3, 4)  # Randomize penalty threshold
        
        self.random_epsilon = np.random.uniform(0,0.5)  # Randomize epsilon for randomization
        self.direction_error = np.random.uniform(0,1)  # Randomize direction error for randomization
        
        self.noise_obs_sigma = np.random.uniform(0, 0.1)  # Add some noise to observation to make it more realistic
        self.noise_action_sigma = np.random.uniform(0,0.1)  # Add some noise to action to make it more realistic
        
        
        
        self.robot_position = np.random.uniform(self.margin, [self.env_width-self.margin,self.env_height-self.margin])  # Randomize robot position
        self.hand_position = np.random.uniform(self.margin, [self.env_width-self.margin,self.env_height-self.margin])  # Randomize hand position
        # self.hand_position = np.clip(self.hand_position, self.margin, self.grid_size-self.margin)  # Ensure hand stays within grid bounds
        
        # self.hand_move_mode = 'random' if np.random.rand() < 0.1 else 'towards_robot'  # Randomize hand movement mode
        # self.hand_move_mode = 'towards_robot'
        
        self.current_distance = np.linalg.norm(self.robot_position - self.hand_position)
        self.pre_distance = self.current_distance
        self.last_action = np.zeros(2)
        self.steps = 0
        self.trajectory_points = [self.robot_position.copy()] # New: Reset trajectory and add initial position
        
        self.fixed_point = np.array([self.grid_size*random.uniform(0.2,1.3),self.grid_size])
        
        vec_arm = self.fixed_point - self.hand_position
        unit_vec_arm = self.safe_normalize(vec_arm)
        self.blocking_point = self.hand_position + unit_vec_arm * self.arm_blocking_length
        if self.blocking_point[1]>self.env_height:
            self.blocking_point = self.fixed_point.copy()
        

        for _ in range(8):
            self.hand_history_buffer.append(np.zeros(2))
        
        
        return self._get_obs(), self._get_info()

    def _reward(self,action):
        terminated = False
        truncated = False
        reward = 0  # Initialize reward
        done_reason = None  # Initialize done reason



        # self.dist_arm = self.dist_point_to_segment_correct(self.robot_position,self.hand_position, self.fixed_point)[0]
        if self.dist_arm < self.distance_threshold_arm:
            reward += self.reward_arm 
            terminated = True  # Truncate if arm is too short

        # boundary penalty
        if np.any(self.robot_position <= self.margin) or (self.env_height-self.robot_position[1] <=self.margin) or self.env_width-self.robot_position[0] <=self.margin:
            reward += self.reward_bound
            terminated = True  # Truncate if robot goes out of bounds
            done_reason = "out of bounds"

    
        # Auxiliary Rewards -  distance to hand
        self.current_distance = np.linalg.norm(self.robot_position - self.hand_position)
        self.distance.append(self.current_distance)
        reward += (self.current_distance-self.pre_distance)*self.distance_reward_factor  # Reward shaping based on distance change
        self.pre_distance = self.current_distance

        # Obstacle handling with two thresholds
        if self.current_distance < self.distance_threshold_collision:
            reward += self.reward_hand
            terminated = True  # Terminate if too close to obstacles
            done_reason = "collision with obstacle"
        elif self.current_distance < self.distance_threshold_penalty:
            reward -= self.penalty_factor * (self.distance_threshold_penalty - self.current_distance)  # Penalty for being too close to obstacles

        reward -= self.smooth_action_penalty * np.linalg.norm(action - self.last_action)

        # Small reward for each step taken to encourage exploration
        reward+= self.reward_step 

        # Truncate if max steps reached and give max step reward
        if self.steps >= self.max_steps:
            reward += self.reward_max_step
            truncated = True  

        return reward,terminated,truncated,done_reason

    def _get_hand_movement(self):

        # if self.hand_move_mode == 'random':
        #     move_hand = np.random.uniform(-1, 1, size=2)  # Randomly move the hand position slightly
        # elif self.hand_move_mode == 'towards_robot':
        #     dir_vector = self.robot_position - self.hand_position
        #     if np.linalg.norm(dir_vector) > 0:
        #         dir_vector /= np.linalg.norm(dir_vector)
        #     move_hand = dir_vector * self.stride_hand  # Move hand towards robot position
        if random.random() < self.hand_move_epsilon:
            move_hand = np.random.uniform(-1, 1, size=2)  # Randomly move the hand position slightly
        else:
            dir_vector = self.robot_position - self.hand_position
            if np.linalg.norm(dir_vector) > 0:
                dir_vector /= np.linalg.norm(dir_vector)
            move_hand = dir_vector * self.stride_hand  # Move hand towards robot position
        
        return move_hand
    


    def _check_line_intersection(self, p1, p2, p3, p4):
        """
        判断线段 p1-p2 (机器人路径) 和 p3-p4 (手臂) 是否相交
        """
        def ccw(A, B, C):
            # 判断三个点的方向 (Counter-Clockwise)
            return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])

        # 如果 p1-p2 的两个端点在 p3-p4 两侧，且 p3-p4 的两个端点在 p1-p2 两侧，则相交
        return ccw(p1, p3, p4) != ccw(p2, p3, p4) and ccw(p1, p2, p3) != ccw(p1, p2, p4)




    def step(self, action):
        # --- 1. 手掌移动逻辑 (保持不变) ---
        if self.random:
            action += np.random.normal(0, self.noise_action_sigma, size=self.action_space.shape)
        
        if random.random() < self.random_epsilon:
            action = np.random.uniform(-1, 1, size=self.action_space.shape)
            move_hand = action * self.stride_hand
        else:
            # angle = 0.1 # 简化了你的原始代码
            # R = np.array([
            #     [np.cos(angle), -np.sin(angle)],
            #     [np.sin(angle),  np.cos(angle)]
            # ])
            # move_hand = R @ (action * self.stride_hand)
            move_hand = action * self.stride_hand
        
        bio_cost = self._calculate_biomechanical_cost(self.hand_position, move_hand)
        
        move_hand = self.execution_filter.apply(move_hand, self.hand_position)  
        self.hand_position += move_hand
        self.hand_position = np.clip(self.hand_position, self.margin, [self.env_width-self.margin, self.env_height-self.margin])
        
        self.hand_history_buffer.append(move_hand)
        # --- 2. 机器人移动逻辑 (修改部分) ---
        
        # [关键步骤 A]：在移动前记录机器人旧位置
        old_robot_position = self.robot_position.copy()

        action_robot, _ = self.robot_model.predict(self._get_obs_robot())
        self.last_action_robot = action_robot.copy()
        
        # 更新机器人位置
        self.robot_position += action_robot * self.stride_robot
        self.trajectory_points.append(self.robot_position.copy())
        self.steps += 1
        
        # 更新手臂固定点 (确保这是最新的手臂位置)
        self.calculate_fix_point()
        vec_arm = self.fixed_point - self.hand_position
        unit_vec_arm = self.safe_normalize(vec_arm)
        self.blocking_point = self.hand_position + unit_vec_arm * self.arm_blocking_length
        if self.blocking_point[1]>self.env_height:
            self.blocking_point = self.fixed_point.copy()
        
        # --- 3. 奖励计算与状态判断 ---
        reward, terminated, truncated, done_reason = self._reward(action)
        reward -= bio_cost

        # [关键步骤 B]：碰撞检测逻辑
        # 线段1: 机器人路径 (old_robot_position -> self.robot_position)
        # 线段2: 手臂障碍 (self.fix_point -> self.hand_position)
        
        if self._check_line_intersection(old_robot_position, self.robot_position, self.fixed_point, self.hand_position):
            terminated = True         # 强制结束回合
            reward -= 0          # [可选] 给予较大的碰撞惩罚
            done_reason = "collision_arm" # 更新结束原因
            # print("Robot hit the arm!")

        self.last_action = action.copy()
        info = self._get_info()
        info['done_reason'] = done_reason
        info['distance_mean'] = np.mean(self.distance)
        observation = self._get_obs()
        
        return observation, reward, terminated, truncated, info

    def render(self, mode="human"):
 
        pygame.display.init()
        self.window = pygame.display.set_mode(
                (int(self.grid_size * self.cell_size), int(self.grid_size * self.cell_size))
            )
        pygame.display.set_caption("CustomEnv")
        if self.clock is None:
            self.clock = pygame.time.Clock()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                import sys
                sys.exit() # Exit the program

            elif event.type == pygame.MOUSEBUTTONDOWN:
                mouse_x, mouse_y = event.pos
                self.hand_position = np.array([mouse_x/self.cell_size, mouse_y/self.cell_size])

            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:  # 空格键切换暂停
                    self.pause = not self.pause


        canvas = pygame.Surface((self.grid_size * self.cell_size, self.grid_size * self.cell_size))
        canvas.fill(WHITE)
        virus_image = pygame.image.load("hand.png").convert_alpha()  # Load an image if needed, but not used here
        robot_image = pygame.transform.scale(virus_image, (int(self.cell_size * 2), int(self.cell_size * 2)))  # Scale the image
        # New: Draw the trajectory
        if len(self.trajectory_points) > 1:
            scaled_points = []
            for point in self.trajectory_points:
                scaled_points.append((int(point[0] * self.cell_size), int(point[1] * self.cell_size)))
            
            # Draw lines between consecutive points
            pygame.draw.lines(canvas, BLUE, False, scaled_points, 2) # Blue line, not closed, 2 pixels wide
            
            # Optionally, draw small circles at each point to emphasize
            for point_coord in scaled_points:
                pygame.draw.circle(canvas, BLUE, point_coord, 3) # Small blue circles

        # Draw robot
        pygame.draw.circle(
            canvas,
            RED,
            (int(self.robot_position[0] * self.cell_size), int(self.robot_position[1] * self.cell_size)),
            int(self.cell_size * 0.2)
        )
        # Draw obstacles

        canvas.blit(robot_image, (int((self.hand_position[0]-1) * self.cell_size), int((self.hand_position[1]-1) * self.cell_size+1)))
        pygame.draw.circle(canvas,
                            GREEN, 
                            (int((self.hand_position[0]) * self.cell_size), 
                            int((self.hand_position[1]) * self.cell_size+1)), 
        int(self.cell_size * 0.2)
        )

        self.window.blit(canvas, canvas.get_rect())
        pygame.event.pump()
        pygame.display.flip()
        self.clock.tick(self.metadata["render_fps"])
        time.sleep(0.5)
    
    def load_args(self, args):
        pass

    def save_args(self,path):
        env_args = {
            "grid_size": self.grid_size,
            "distance_threshold_penalty":self.distance_threshold_penalty,
            "distance_threshold_collision":self.distance_threshold_collision,
            "penalty_factor":self.penalty_factor,
            "distance_reward_factor":self.distance_reward_factor,
            "smooth_action_penalty":self.smooth_action_penalty,
            "max_steps":self.max_steps,
            "margin":self.margin,
            "reward_step":self.reward_step,
            "reward_max_step":self.reward_max_step,
            "reward_bound":self.reward_bound,
            "reward_arm":self.reward_arm,
            "reward_hand":self.reward_hand,
            "stride_robot_range":self.stride_robot_random,
            "stride_hand_range":self.stride_hand_random,
            "move_hand_epsilon":self.hand_move_epsilon,



        }
        with open(os.path.join(path, "env_args.json"), "w") as f:
            json.dump(env_args, f,indent=4)
        

    def close(self):
        pygame.display.quit()
        pygame.quit()





In [6]:
def render_environment(robot_position, hand_position, fix_point,trajectory_points, grid_size=10, cell_size=50):
    WHITE = (255, 255, 255)
    RED = (255, 0, 0)
    GREEN = (0, 255, 0)
    BLUE = (0, 0, 255)
    # print(robot_position, hand_position)
    pygame.init()
    window = pygame.display.set_mode((grid_size * cell_size*1.5, grid_size * cell_size))
    canvas = pygame.Surface((grid_size * cell_size*1.5, grid_size * cell_size))
    canvas.fill(WHITE)
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            import sys
            sys.exit() # Exit the program

        elif event.type == pygame.MOUSEBUTTONDOWN:
            mouse_x, mouse_y = event.pos
            hand_position = np.array([mouse_x/cell_size, mouse_y/cell_size])

    if len(trajectory_points) > 1:
        scaled_points = [(int(point[0] * cell_size), int(point[1] * cell_size)) for point in trajectory_points]
        pygame.draw.lines(canvas, BLUE, False, scaled_points, 2)
        for point_coord in scaled_points:
            pygame.draw.circle(canvas, BLUE, point_coord, 3)

    pygame.draw.lines(canvas, (255, 224, 189), False,[hand_position*cell_size, [fix_point[0]*cell_size,fix_point[1]*cell_size]],width=25)

    virus_image = pygame.image.load("../hand.png").convert_alpha()  # Load an image if needed, but not used here
    robot_image = pygame.transform.scale(virus_image, (int(cell_size * 2), int(cell_size * 2)))  # Scale the 
    pygame.draw.circle(canvas, RED, (int(robot_position[0] * cell_size), int(robot_position[1] * cell_size)), int(cell_size * 0.2))
    pygame.draw.circle(canvas, GREEN, (int(hand_position[0] * cell_size), int(hand_position[1] * cell_size)), int(cell_size * 0.2))
    canvas.blit(robot_image, (int((hand_position[0]-1) * cell_size), int((hand_position[1]-1) * cell_size)))
    font = pygame.font.Font(None, 24)
    text = font.render(f"{hand_position[0]},{hand_position[1]}", True, BLUE)
    text_rect = text.get_rect()
    text_rect.center = (int(hand_position[0] * cell_size), int(hand_position[1] * cell_size))
    window.blit(canvas, canvas.get_rect())
    # window.blit(text, text_rect)
    pygame.display.flip()

    return hand_position

In [7]:
from collections import deque
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback, CallbackList
class DebugCallback(BaseCallback):
    def __init__(self, env, render_freq=10000, n_episodes=1, log_freq=10000, verbose=1):
        super().__init__(verbose)
        self.log_freq = log_freq
        # 用deque统计最近N个done的终止原因，避免内存爆炸
        self.termination_reasons = deque(maxlen=1000)  # 统计最近1000次终止
        self.env_to_render = env
        self.render_freq = render_freq
        self.n_episodes = n_episodes
        self.distance_mean = deque(maxlen=1000) 

    def _on_step(self) -> bool:
        # 先从env info里读取终止原因
        # print((self.locals.keys()))
        infos = self.locals.get('infos', None)

        dones = self.locals.get('dones', None)
        if infos is not None and dones is not None:
            for done, info in zip(dones, infos):
                if done and info is not None and 'done_reason' in info:
                    self.termination_reasons.append(info['done_reason'])
                    self.distance_mean.append(info['distance_mean'])

        # 每log_freq步打印信息
        # if self.num_timesteps % self.render_freq == 0 and self.verbose:
        #     for ep in range(self.n_episodes):
        #         obs = self.env_to_render.reset()
        #         done = False
        #         while not done:
        #             action, _states = self.model.predict(obs, deterministic=True)
        #             obs, rewards, done, info = self.env_to_render.step(action)
        #             self.env_to_render.render()
        #             time.sleep(0.6)

        #             if done:

        #                 self.env_to_render.close()
        #                 break





        if self.num_timesteps % self.log_freq == 0 and self.verbose:
            log = self.model.logger.name_to_value
            ep_rew = log.get('rollout/ep_rew_mean', None)
            ep_len = log.get('rollout/ep_len_mean', None)
            loss = log.get('train/loss', None)
            v_loss = log.get('train/value_loss', None)
            p_loss = log.get('train/policy_gradient_loss', None)
            ent_loss = log.get('train/entropy_loss', None)
            kl = log.get('train/approx_kl', None)

            # 统计终止原因比例
            total = len(self.termination_reasons)
            if total > 0:
                count_hand = sum(1 for r in self.termination_reasons if r == 'out of bounds')
                ratio_hand = count_hand / total
            else:
                ratio_hand = 0.0
            distance_mean = sum(self.distance_mean) / len(self.distance_mean) if len(self.distance_mean) > 0 else 0.0

            # print(f"[{self.num_timesteps:7d}] ep_rew_mean={ep_rew}, ep_len_mean={ep_len}, loss={loss:.3f}, "
            #       f"v_loss={v_loss:.3f}, p_loss={p_loss:.3f}, ent_loss={ent_loss:.3f}, kl={kl:.4f}, "
            #       f"termination_reason_hand_ratio={ratio_hand:.3f}")
            self.logger.record("custom/termination_reason_ratio", ratio_hand)

            self.logger.record("custom/distance_mean", distance_mean)
            self.logger.dump(step=self.num_timesteps)

        return True


In [7]:
# from matplotlib import pyplot as plt

# def metric(trajectory):
#     """
#     trajectory: list of tuples, each tuple contains (observation, action, hand_movement, reward)
#     """
    
#     if not isinstance(trajectory, list):
#         raise TypeError("trajectory should be a list of tuples")

#     # distance = [x[6] for x in trajectory]
#     # return distance
    
#     # plt.hist(distance, bins=10, density=True,edgecolor='black', alpha=0.7,color='skyblue')
#     # plt.xlabel('distance')
#     # plt.ylabel('density')
#     # plt.title('distance distribution')
#     # plt.pause(0.1)

#     # distance_aproximity
#     for item in trajectory:
#         obs = item[0]
#         distance = obs[0:2]




#     # direction_alignment
#     for item in trajectory:
#         obs = item[0]
#         direction = 
#         hand_movement = item[2]

#         position_robot,position_hand = obs[0:2],obs[2:4]
#         direction = position_robot - position_hand
#         consine_angle = np.dot(direction, hand_movement) / (np.linalg.norm(direction) * np.linalg.norm(hand_movement))
#         angle = np.arccos(consine_angle)

#     # reaction_time

    



In [24]:
import torch as th
import torch.nn as nn
import gymnasium as gym
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class SlicingFeatureExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box):
        # 假设最终输出维度是 64
        super().__init__(observation_space, features_dim=64)
        
        # --- 配置参数 (硬编码或通过参数传入) ---
        self.scalar_dim = 14       # 标量特征长度
        self.history_len = 8      # 时间步长
        self.history_channels = 2  # dx, dy
        
        # --- 1. 1D CNN 处理历史 ---
        self.cnn_net = nn.Sequential(
            # 输入形状: (Batch, 2, 10)
            nn.Conv1d(2, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(output_size=1), # (Batch, 32, 1)
            nn.Flatten() # (Batch, 32 * 8) 
        )
        
        # 计算 CNN 输出大小
        # 简单的 Conv1d (padding=1, stride=1) 不改变长度，所以是 32 * 10
        self.cnn_output_dim = 32 
        self.scalar_output_dim = 16 
        self.output_dim = 64
        # --- 2. 标量处理 (可选) ---
        # 简单的线性层处理标量
        self.scalar_net = nn.Sequential(
            nn.Linear(self.scalar_dim,self.scalar_output_dim),
            nn.ReLU()
        )
        
        # --- 3. 融合层 ---
        # 把 CNN特征 + 标量特征 融合
        self.fusion_net = nn.Sequential(
            nn.Linear(self.cnn_output_dim + self.scalar_output_dim, self.output_dim),
            nn.ReLU()
        )

    def forward(self, observations: th.Tensor) -> th.Tensor:
        # observations shape: (Batch, 34)
        
        # --- 关键步骤：手动切片 (Slicing) ---
        # 假设前 14 个是标量，后 20 个是历史
        # 注意：这要求你的 Env _get_obs 拼接顺序必须严格一致！
        scalar_part = observations[:, :self.scalar_dim]       # (Batch, 14)
        history_part = observations[:, self.scalar_dim:]      # (Batch, 20)
        
        # --- 关键步骤：重塑 (Reshape) ---
        # 将扁平的历史 (Batch, 20) 变回 (Batch, 2, 10) 给 CNN 用
        # view 的参数: (Batch, Channels, Length)
        history_reshaped = history_part.view(-1, self.history_channels, self.history_len)
        
        # --- 1. CNN 流 ---
        cnn_out = self.cnn_net(history_reshaped)
        
        # --- 2. 标量 流 ---
        scalar_out = self.scalar_net(scalar_part)
        
        # --- 3. 拼接 ---
        combined = th.cat([cnn_out, scalar_out], dim=1)
        
        # --- 4. 融合输出 ---
        return self.fusion_net(combined)

In [25]:
import gymnasium as gym
import torch
from stable_baselines3 import PPO,SAC
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecMonitor
from stable_baselines3.common.monitor import Monitor
# from stable_baselines3.commom.buffers import ReplayBuffer
from stable_baselines3.common.logger import configure
from stable_baselines3.common.utils import safe_mean


save_path = "logs/robot/"
i=1
while os.path.exists(save_path+str(i)):
    i+=1
save_path = f"{save_path}{i}"
os.makedirs(save_path)

tensorboard_dir = "./sac_custom_env_tensorboard"
os.makedirs(tensorboard_dir, exist_ok=True)
tensorboard_log_dir = tensorboard_dir + '/'+ f'best_model_sac{i}'
load_model = "logs1/best_model_sac21/best_model.zip"


policy_kwargs = dict(
    net_arch=[128,256,128],
    features_extractor_class=SlicingFeatureExtractor,
    features_extractor_kwargs=dict(),
    # activation_fn=torch.nn.ReLU  # 改为 ReLU，通常更适合稀疏奖励
)
# hand_model = SAC.load('logs_hand/best_model_sac17/best_model.zip')
env_raw = RehabilitationEnv(training_mode='robot',hand_model=None)
env = DummyVecEnv([lambda: Monitor(env_raw)])
model_robot = SAC("MlpPolicy", env, verbose=1,ent_coef='auto',policy_kwargs=policy_kwargs,tensorboard_log=tensorboard_log_dir)
# model_robot = SAC.load('logs1/best_model_sac23/best_model.zip',env=env,tensorboard_log=tensorboard_log_dir)


eval_callback = EvalCallback(
    env,
    best_model_save_path=save_path,
    log_path = './logs/',
    eval_freq=10000,  # 每1000步评估一次
    deterministic=True,
    render=True,
    n_eval_episodes=10,  # 每次评估5个episode
)



# debug_callback = DebugCallback(env=env,log_freq=10000, verbose=1)
# callback = CallbackList([eval_callback, debug_callback])
callback = eval_callback

model_robot.learn(total_timesteps=500000, callback=callback)

settings = {

    'tensorboard_log' :tensorboard_log_dir,
           }
# env_raw.save_args(save_path)
# with open(os.path.join(save_path, "settings.json"), "w") as f:
#     json.dump(settings, f)

env.close()


Using cpu device
Logging to ./sac_custom_env_tensorboard/best_model_sac1\SAC_1


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 2.5      |
|    ep_rew_mean     | -123     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 1249     |
|    time_elapsed    | 0        |
|    total_timesteps | 10       |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 2.12     |
|    ep_rew_mean     | -137     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 1214     |
|    time_elapsed    | 0        |
|    total_timesteps | 17       |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 2.33     |
|    ep_rew_mean     | -140     |
| time/              |          |
|    episodes        | 12       |
|    fps             | 1473     |
|    time_elapsed    | 0        |
|    total_timesteps | 28       |
--------------

g:\anaconda\envs\RL\lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:259: UserWarning: You tried to call render() but no `render_mode` was passed to the env constructor.
  warnings.warn("You tried to call render() but no `render_mode` was passed to the env constructor.")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 6.13     |
|    ep_rew_mean     | -102     |
| time/              |          |
|    episodes        | 1820     |
|    fps             | 44       |
|    time_elapsed    | 225      |
|    total_timesteps | 10014    |
| train/             |          |
|    actor_loss      | 93.6     |
|    critic_loss     | 73.9     |
|    ent_coef        | 0.107    |
|    ent_coef_loss   | -0.237   |
|    learning_rate   | 0.0003   |
|    n_updates       | 9913     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 6.06     |
|    ep_rew_mean     | -101     |
| time/              |          |
|    episodes        | 1824     |
|    fps             | 44       |
|    time_elapsed    | 226      |
|    total_timesteps | 10041    |
| train/             |          |
|    actor_loss      | 93.2     |
|    critic_loss     | 103      |
|    ent_coef 

In [63]:
import gymnasium as gym
import torch
from stable_baselines3 import PPO,SAC
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecMonitor
from stable_baselines3.common.monitor import Monitor
# from stable_baselines3.commom.buffers import ReplayBuffer
from stable_baselines3.common.logger import configure
from stable_baselines3.common.utils import safe_mean


save_path = "logs/hand_stage2/"
i=1
while os.path.exists(save_path+str(i)):
    i+=1
save_path = f"{save_path}{i}"
os.makedirs(save_path)

tensorboard_dir = "./sac_tensorboard_hand/"
os.makedirs(tensorboard_dir, exist_ok=True)
tensorboard_log_dir = tensorboard_dir + '/'+ f'best_model_sac{i}'



policy_kwargs = dict(
    net_arch=[128,256,128],
    features_extractor_class=SlicingFeatureExtractor,
    features_extractor_kwargs=dict(),
    # activation_fn=torch.nn.ReLU  # 改为 ReLU，通常更适合稀疏奖励
)
# robot_model = SAC.load('logs/robot/1/best_model.zip')
hand_model = SAC.load('logs/hand/1/best_model.zip')
env_raw = RehabilitationEnv(training_mode='robot',hand_model=hand_model)
env = DummyVecEnv([lambda: Monitor(env_raw)])
# hand_model = SAC("MlpPolicy", env, verbose=1,ent_coef='auto',policy_kwargs=policy_kwargs,tensorboard_log=tensorboard_log_dir)
robot_model = SAC.load('logs/robot/1/best_model.zip',env=env)


training_model = robot_model

eval_callback = EvalCallback(
    env,
    best_model_save_path=save_path,
    log_path = './logs/',
    eval_freq=10000,  # 每1000步评估一次
    deterministic=True,
    render=True,
    n_eval_episodes=10,  # 每次评估5个episode
)



# debug_callback = DebugCallback(env=env,log_freq=10000, verbose=1)
# callback = CallbackList([eval_callback, debug_callback])
callback = eval_callback

training_model.learn(total_timesteps=500000, callback=callback)

settings = {

    'tensorboard_log' :tensorboard_log_dir,
           }
# env_raw.save_args(save_path)
# with open(os.path.join(save_path, "settings.json"), "w") as f:
#     json.dump(settings, f)

env.close()


Logging to ./sac_custom_env_tensorboard/best_model_sac1\SAC_5
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.25     |
|    ep_rew_mean     | -171     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 684      |
|    time_elapsed    | 0        |
|    total_timesteps | 13       |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.38     |
|    ep_rew_mean     | -182     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 870      |
|    time_elapsed    | 0        |
|    total_timesteps | 27       |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.42     |
|    ep_rew_mean     | -163     |
| time/              |          |
|    episodes        | 12       |
|    fps             | 819      |
|    time_elapsed   

KeyboardInterrupt: 

In [30]:
x =deque(maxlen=1)
x.append(1)
print(len(x))
x.append(2)
print(len(x))

1
1


In [104]:
from stable_baselines3 import SAC,PPO
# env =DummyVecEnv([lambda: Monitor(CustomEnv())])  # Wrap the environment with Monitor for logging

# env = VecNormalize.load('vec_norm.pkl',env)
# env = VecNormalize(env)  # Apply normalization to the environment
# seed = np.random.randint(0,1000)
# from custom_env import CustomEnv
# env = CustomEnv_hand()
pygame.init()
grid_s = 10
cell_s = 50
screen = pygame.display.set_mode((int(grid_s * cell_s * 1.5), int(grid_s * cell_s)))
# env.random = False

# env = CustomEnv()
# model_robot= SAC.load("logs/best_model_sac88/best_model.zip",env=env)  # Load the best model

model_hand = SAC.load("logs/hand/1/best_model.zip")  # Load the best model
# # ['healthy', 'parkinson', 'stroke', 'ataxia']
# env = CustomEnv_hand(robot_model=model_robot,hand_mode='stroke')
model_robot = SAC.load("logs/robot_stage2/1/best_model.zip")  # Load the best model
env = RehabilitationEnv(training_mode='robot',robot_model=model_robot,hand_model=model_hand)
env.arm_blocking_length = 4
obs = env.reset()
env.random_epsilon = 0.1
env.arm_blocking_ratio = 0.5
env.stride_hand = 1.5
env.stride_robot = 2
env.distance_threshold_collision = 1.5
state_history = []
max_steps = 40000
env.bio_filter.mode = 'healthy'
for i in range(max_steps):
    obs = env._get_obs()
    info = env._get_info()
    action, _states = model_robot.predict(obs, deterministic=True)

    obs, reward, teminated,_, info = env.step(action)
    state_history.append(obs)
    print(f"obs:{obs}")
    print(f"action:{action}")

    print('blocking point',env.blocking_point)

    # print(info["robot_position"],info["hand_position"])
    # print("Reward:", reward)
    # print("distance_arm:",info['distance_arm'])
    # env.render()
    render_aesthetic(info["robot_pos"], info["hand_pos"],info["fixed_point"], info["blocking_point"], trajectory_points=env.trajectory_points,window=screen)
    time.sleep(0.5)  # Control the frame rate
    if teminated:
        render_aesthetic(info["robot_pos"], info["hand_pos"],info["fixed_point"], info["blocking_point"], trajectory_points=env.trajectory_points,window=screen)

        # render_aesthetic(info["robot_position"], info["hand_position"],info["fixed_point"], trajectory_points=env.trajectory_points)
        # env.render()
        # time.sleep(0.6)
        env.close()
        break
        print("Resetting environment")
# metric(state_history)

obs:[ 7.9278517e+00  2.1688991e+00  1.2599999e+01  3.2488201e+00
  4.6928749e+00  7.9219656e+00  7.1185021e+00  2.3134353e+00
  7.6866813e+00  1.9636110e+00  7.4602804e+00  1.0059069e+01
  1.0135391e+01  6.5549440e+00 -7.6434284e-02 -8.5503161e-02
  3.4866489e-02 -2.8971429e-03 -4.4159580e-02 -2.8755285e-02
  3.9730966e-02  6.0541410e-02  7.9598173e-02 -5.9274711e-02
  3.7418313e-02 -3.3096924e-02 -2.0109147e-03  4.2114478e-02
 -1.3719935e+00 -5.0817603e-01]
action:[0.56835616 0.7050647 ]
blocking point [10.09974255  6.56806055]
obs:[ 6.1986303e+00  3.8889456e+00  1.1046175e+01  2.9847794e+00
  4.8754163e+00  6.3022676e+00  8.6952600e+00  3.8834445e+00
  6.1473155e+00  1.9643953e+00  7.5575027e+00  9.9509201e+00
  9.2338686e+00  6.5793996e+00 -1.3514018e-02  9.3504656e-03
  9.6410522e-03  3.5676945e-02  5.0728939e-02  5.2765701e-02
 -3.1855609e-02 -3.7858024e-02  6.6496432e-03  1.7446758e-02
  9.5544802e-03  3.8988933e-02 -1.4218655e+00 -6.3416350e-01
 -1.4485039e+00 -3.0715549e-01]
ac

In [78]:
import math
import pygame
import pygame.gfxdraw
import numpy as np

def draw_capsule_rotated(surf, color, center_x, center_y, width, length, angle_deg):
    """
    画一个胶囊：
    angle_deg 是胶囊延伸的方向。
    """
    rad = math.radians(angle_deg)
    cos_a = math.cos(rad)
    sin_a = math.sin(rad)

    # 胶囊底部中心 (center_x, y)
    # 胶囊顶部中心 (沿着 angle 方向延伸 length 长度)
    x2 = center_x + length * cos_a
    y2 = center_y + length * sin_a

    # 计算垂直于延伸方向的宽度向量
    dx = (width / 2) * math.sin(rad)
    dy = (width / 2) * math.cos(rad)

    # 4个角点
    points = [
        (center_x - dx, center_y + dy), # 左底
        (center_x + dx, center_y - dy), # 右底
        (x2 + dx, y2 - dy),             # 右顶
        (x2 - dx, y2 + dy)              # 左顶
    ]
    
    pygame.gfxdraw.aapolygon(surf, points, color)
    pygame.gfxdraw.filled_polygon(surf, points, color)
    
    # 两个圆头
    pygame.gfxdraw.aacircle(surf, int(center_x), int(center_y), int(width/2), color)
    pygame.gfxdraw.filled_circle(surf, int(center_x), int(center_y), int(width/2), color)
    pygame.gfxdraw.aacircle(surf, int(x2), int(y2), int(width/2), color)
    pygame.gfxdraw.filled_circle(surf, int(x2), int(y2), int(width/2), color)

def draw_detailed_hand(surf, fill_color, border_color, center, radius, angle_deg=0):
    """
    绘制一个结构清晰的手掌（带指甲，明确表示手背向上）
    """
    cx, cy = center
    base_rad = math.radians(angle_deg)
    
    # 指甲颜色：比肤色更淡、更白一点
    # 假设 fill_color 是 (255, 204, 188)，指甲可以用 (255, 240, 230)
    nail_color = (min(fill_color[0]+20, 255), min(fill_color[1]+20, 255), min(fill_color[2]+20, 255))
    knuckle_color = border_color # 关节用深色
    
    def get_rotated_offset(ox, oy):
        """将相对坐标 (ox, oy) 旋转并叠加到中心"""
        rx = ox * math.cos(base_rad) - oy * math.sin(base_rad)
        ry = ox * math.sin(base_rad) + oy * math.cos(base_rad)
        return cx + rx, cy + ry

    # --- 尺寸调整 ---
    palm_size = radius * 1.15
    finger_width = radius * 0.48
    finger_len = radius * 1.5
    
    # --- 定义手指配置 ---
    # (x偏移, y偏移, 相对角度, 长度系数)
    fingers = [
        (radius*0.15,  -radius*0.55, -12, 0.9),  # 食指 (略短)
        (radius*0.25,   0,           0,   1.0),  # 中指 (最长)
        (radius*0.15,   radius*0.55,  12,  0.9), # 无名指 (略短)
        (radius*0.05,   radius*1.0,   25,  0.75) # 小指 (最短, 新增) - 显得更真实
    ]
    
    # 1. 先画手指轮廓 (Border)
    for fx, fy, fang, flen in fingers:
        pos = get_rotated_offset(fx, fy)
        draw_capsule_rotated(surf, border_color, pos[0], pos[1], finger_width+4, finger_len*flen, angle_deg + fang)
    
    # 2. 画大拇指轮廓
    # 大拇指位置调整：手背向上时，大拇指根部其实在手掌侧面偏里
    thumb_pos = get_rotated_offset(-radius*0.2, -radius * 0.6)
    thumb_angle = -55 
    draw_capsule_rotated(surf, border_color, thumb_pos[0], thumb_pos[1], finger_width*1.3+4, finger_len*0.85, angle_deg + thumb_angle)

    # 3. 画掌心轮廓
    pygame.gfxdraw.aacircle(surf, int(cx), int(cy), int(palm_size), border_color)
    pygame.gfxdraw.filled_circle(surf, int(cx), int(cy), int(palm_size), border_color)

    # ==========================
    #       填充内部 (Fill)
    # ==========================
    
    # 辅助函数：画指甲
    def draw_nail(start_x, start_y, width, length, angle):
        # 指甲位置：在手指末端 80% 处
        nail_dist = length * 0.75
        nail_w = width * 0.6
        nail_h = width * 0.5 # 指甲稍微方一点
        
        rad = math.radians(angle)
        # 计算指甲中心
        nx = start_x + nail_dist * math.cos(rad)
        ny = start_y + nail_dist * math.sin(rad)
        
        # 这里简单画个小圆或胶囊当指甲
        draw_capsule_rotated(surf, nail_color, nx, ny, nail_h, nail_w * 0.2, angle)

    # 4. 填充手指 + 画指甲
    for fx, fy, fang, flen in fingers:
        pos = get_rotated_offset(fx, fy)
        current_len = finger_len * flen
        current_angle = angle_deg + fang
        
        # 填充肤色
        draw_capsule_rotated(surf, fill_color, pos[0], pos[1], finger_width, current_len, current_angle)
        
        # 画指甲 (关键！)
        draw_nail(pos[0], pos[1], finger_width, current_len, current_angle)
        
        # 画指关节 (Knuckles) - 在手指根部画一条淡淡的弧线或圆点
        # 这里简单用一个小圆点表示指关节隆起
        pygame.gfxdraw.aacircle(surf, int(pos[0]), int(pos[1]), int(finger_width*0.4), (230, 150, 130)) # 稍微深一点的肤色
        pygame.gfxdraw.filled_circle(surf, int(pos[0]), int(pos[1]), int(finger_width*0.4), (230, 150, 130))

    # 5. 填充大拇指 + 指甲
    draw_capsule_rotated(surf, fill_color, thumb_pos[0], thumb_pos[1], finger_width*1.3, finger_len*0.85, angle_deg + thumb_angle)
    draw_nail(thumb_pos[0], thumb_pos[1], finger_width*1.3, finger_len*0.85, angle_deg + thumb_angle)

    # 6. 填充掌心
    pygame.gfxdraw.aacircle(surf, int(cx), int(cy), int(palm_size-2), fill_color)
    pygame.gfxdraw.filled_circle(surf, int(cx), int(cy), int(palm_size-2), fill_color)
    
    # 7. 手背特征：掌骨线 (Metacarpal lines)
    # 在手背画两条淡淡的线，模拟肌腱，增加立体感
    for i in [-1, 1]:
        start = get_rotated_offset(-radius*0.5, i * radius*0.3)
        end = get_rotated_offset(0, i * radius*0.2)
        pygame.draw.line(surf, (230, 150, 130), start, end, 2)

# --- 1. 定义学术风格配色 (Scientific Color Palette) ---

COLORS = {
    'bg_main': (250, 250, 250),      
    'grid': (230, 230, 230),         
    
    # === 修改部分：手臂颜色 (红色系) ===
    # 1. 阻挡段 (实心，深色，不可穿越)
    'arm_fill': (255, 204, 188),     # 正常的柔和肤色
    'arm_border': (230, 74, 25),     # 深砖红色边缘，强调实体感
    
    # 2. 安全段 (虚像，极淡粉色，可穿越)
    # 比背景稍红一点点，像褪色或半透明的效果
    'arm_safe_fill': (255, 240, 235), # 极淡的粉白
    'arm_safe_border': (240, 180, 170), # 淡珊瑚色边缘，柔和不刺眼
    # ================================

    'robot': (231, 76, 60),          # 扁平红
    'robot_shadow': (200, 50, 50),   
    'hand_core': (46, 204, 113),     
    'trajectory': (52, 152, 219),    
    'text': (50, 60, 80)             
}

def draw_aa_circle(surf, color, center, radius):
    """画抗锯齿的实心圆"""
    x, y = int(center[0]), int(center[1])
    pygame.gfxdraw.aacircle(surf, x, y, radius, color)
    pygame.gfxdraw.filled_circle(surf, x, y, radius, color)

def draw_capsule(surf, color, start_pos, end_pos, width):
    """画胶囊形状（用于模拟手臂），比单纯的粗线好看"""
    x1, y1 = start_pos
    x2, y2 = end_pos
    length = np.hypot(x2-x1, y2-y1)
    if length == 0: return

    angle = np.arctan2(y2-y1, x2-x1)
    
    # 计算矩形的四个角
    dx = width/2 * np.sin(angle)
    dy = width/2 * np.cos(angle)
    
    # 胶囊的身体（多边形）
    points = [
        (x1 - dx, y1 + dy),
        (x2 - dx, y2 + dy),
        (x2 + dx, y2 - dy),
        (x1 + dx, y1 - dy)
    ]
    pygame.gfxdraw.aapolygon(surf, points, color)
    pygame.gfxdraw.filled_polygon(surf, points, color)
    
    # 两个端点的圆头
    draw_aa_circle(surf, color, start_pos, int(width/2))
    draw_aa_circle(surf, color, end_pos, int(width/2))

def render_aesthetic(robot_pos, hand_pos, fix_point,split_point, trajectory_points, 
                    grid_size=10, cell_size=50, window=None):
    
    width_px = int(grid_size * cell_size * 1.5)
    height_px = int(grid_size * cell_size)
    
    if window is None:
        if not pygame.get_init(): pygame.init()
        window = pygame.display.set_mode((width_px, height_px))

    canvas = pygame.Surface((width_px, height_px))
    canvas.fill(COLORS['bg_main'])

    # 1. Grid
    for x in range(0, width_px, cell_size):
        pygame.draw.line(canvas, COLORS['grid'], (x, 0), (x, height_px), 1)
    for y in range(0, height_px, cell_size):
        pygame.draw.line(canvas, COLORS['grid'], (0, y), (width_px, y), 1)

    # ---------------- 2. Draw Arm (Two Segments) ----------------
    # 转换坐标
    fix_px = np.array(fix_point) * cell_size
    hand_px = np.array(hand_pos) * cell_size
    split_px = np.array(split_point) * cell_size
    
    # # 计算手臂向量
    # arm_vec = hand_px - fix_px
    # arm_len = np.linalg.norm(arm_vec)
    
    # # 定义“分割点”：假设靠近肩膀的 35% 是安全的（浅色），剩下 65% 是阻挡的（深色）
    # # 你可以根据环境逻辑调整这个 safe_ratio，例如 0.3 或 0.4
    # safe_ratio = 0.35 
    # split_px = fix_px + arm_vec * safe_ratio
    
    arm_width = 40
    
    # --- A. 绘制安全段 (Safe Segment: Shoulder -> Split) ---
    # 这是一个浅色段，表示机器人可以穿过（例如上臂悬空）
    draw_capsule(canvas, COLORS['arm_safe_border'], fix_px, split_px, arm_width + 4)
    draw_capsule(canvas, COLORS['arm_safe_fill'], fix_px, split_px, arm_width)
    
    # --- B. 绘制阻挡段 (Blocked Segment: Split -> Hand) ---
    # 这是一个深色段，表示实体阻挡
    draw_capsule(canvas, COLORS['arm_border'], split_px, hand_px, arm_width + 4)
    draw_capsule(canvas, COLORS['arm_fill'], split_px, hand_px, arm_width)
    
    # 画关节连接点 (Split Joint) 掩盖接缝
    draw_aa_circle(canvas, COLORS['arm_fill'], split_px, int(arm_width/2))
    
    # 画肩膀关节 (Shoulder Joint)
    # draw_aa_circle(canvas, (100, 100, 100), fix_px, 12) # 灰色机械关节感

    # ---------------- 3. Trajectory ----------------
    if len(trajectory_points) > 1:
        recent_points = trajectory_points[-50:] 
        for i in range(len(recent_points) - 1):
            pt1 = (int(recent_points[i][0] * cell_size), int(recent_points[i][1] * cell_size))
            pt2 = (int(recent_points[i+1][0] * cell_size), int(recent_points[i+1][1] * cell_size))
            alpha = int(255 * (i / len(recent_points)))
            draw_aa_circle(canvas, (*COLORS['trajectory'], alpha), pt1, 2)
            if i > len(recent_points) - 15:
                 pygame.draw.line(canvas, COLORS['trajectory'], pt1, pt2, 2)

    # ---------------- 4. Robot ----------------
    robot_px = np.array(robot_pos) * cell_size
    robot_radius = int(cell_size * 0.25)
    
    draw_aa_circle(canvas, (200, 200, 200), robot_px + (3, 3), robot_radius) # Shadow
    draw_aa_circle(canvas, COLORS['robot'], robot_px, robot_radius) # Body
    draw_aa_circle(canvas, (255, 255, 255), robot_px - (robot_radius*0.3, robot_radius*0.3), int(robot_radius*0.3)) # Highlight

    # ---------------- 5. Hand ----------------
    dx = hand_px[0] - fix_px[0]
    dy = hand_px[1] - fix_px[1]
    hand_angle = math.degrees(math.atan2(dy, dx))
    
    hand_size = cell_size * 0.6
    draw_detailed_hand(canvas, COLORS['arm_fill'], COLORS['arm_border'], 
                      hand_px, hand_size, angle_deg=hand_angle)
    
    draw_aa_circle(canvas, (255, 255, 255), hand_px, int(cell_size * 0.08))

    # ---------------- 6. Text ----------------
    font = pygame.font.SysFont("Arial", 18)
    
    window.blit(canvas, (0, 0))
    pygame.display.flip()
    
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            import sys; sys.exit()
    
    return canvas

In [ ]:
# 初始化一次
pygame.init()
grid_s = 10
cell_s = 50
screen = pygame.display.set_mode((int(grid_s * cell_s * 1.5), int(grid_s * cell_s)))

# 模拟循环
running = True
traj = []
while running:
    # ... 你的 RL 逻辑计算 robot_pos, hand_pos ...
    
    # 假设的数据用于测试
    import time
    t = time.time()
    robot_p = [5 + 2*np.sin(t*3), 5 + 2*np.cos(t*3)]
    hand_p = [5 + 3*np.sin(t), 8]
    fix_p = [5, 10] # 屏幕底部中间
    traj.append(robot_p)
    
    # 调用渲染
    render_aesthetic(robot_p, hand_p, fix_p, traj, 
                     grid_size=grid_s, cell_size=cell_s, window=screen)
    
    pygame.time.delay(20)

SystemExit: 